# 1x1-conv-channel-reshape — faded example 3: Reshape a Linear weight into 1x1 conv layout

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `1x1-conv-channel-reshape`. Running the beacon reports progress on the `CNN: 1x1 conv channel-reshape` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1x1 conv channel-reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`1x1-conv-channel-reshape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "1x1-conv-channel-reshape"
DD_SUBTOPIC = "CNN: 1x1 conv channel-reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To convolutionalize a `Linear(C_in, C_out)` head into a 1x1 `Conv2d`, the only nontrivial step is the weight reshape: `linear.weight` of shape `(C_out, C_in)` must become the conv weight shape `(C_out, C_in, 1, 1)` by adding two singleton kernel dims. The bias copies across unchanged.

## Faded exercise 3

### Faded — build a 1x1 conv from a Linear's weights

Implement `conv_from_linear(linear)` for `linear = nn.Linear(5, 3)`. The conv is constructed and the bias is already copied. You must fill in the assignment that copies `linear.weight` into `conv.weight.data` with the correct `(C_out, C_in, 1, 1)` shape (use `.clone()` so autograd graphs stay separate). Running the conv on an image must match a per-pixel application of the original linear.

**Fill in:** the assignment conv.weight.data = linear.weight.data.view(OC, IC, 1, 1).clone().

In [ ]:
import torch.nn as nn

def conv_from_linear(linear):
    OC, IC = linear.weight.shape
    conv = nn.Conv2d(IC, OC, kernel_size=1)
    raise NotImplementedError()  # TODO: conv.weight.data = linear.weight.data.view(OC, IC, 1, 1).clone()
    conv.bias.data = linear.bias.data.clone()
    return conv

t.manual_seed(0)
linear = nn.Linear(5, 3)
conv = conv_from_linear(linear)
x = t.randn(2, 5, 4, 6)
print(tuple(conv(x).shape))


def _test():
    import torch.nn as nn
    t.manual_seed(0)
    linear = nn.Linear(5, 3)
    conv = conv_from_linear(linear)
    assert conv.weight.shape == (3, 5, 1, 1), conv.weight.shape
    x = t.randn(2, 5, 4, 6)
    x_flat = rearrange(x, 'b c h w -> (b h w) c')
    ref = rearrange(linear(x_flat), '(b h w) c -> b c h w', b=2, h=4, w=6)
    assert t.allclose(conv(x), ref, atol=1e-5)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

def conv_from_linear(linear):
    OC, IC = linear.weight.shape
    conv = nn.Conv2d(IC, OC, kernel_size=1)
    conv.weight.data = linear.weight.data.view(OC, IC, 1, 1).clone()
    conv.bias.data = linear.bias.data.clone()
    return conv

t.manual_seed(0)
linear = nn.Linear(5, 3)
conv = conv_from_linear(linear)
x = t.randn(2, 5, 4, 6)
print(tuple(conv(x).shape))
```
</details>